### EDA NOTEBOOK

Exploratory analysis of `data/cleaned_incidents.csv` (output of `scripts/clean_data.py`).

In [ ]:
# --- imports and settings ---
import matplotlib.pyplot as plt
import pandas as pd

CLEANED_CSV = "../data/cleaned_incidents.csv"  # output of scripts/clean_data.py
LOCAL_TZ = "America/New_York"               # Philadelphia's time zone

# chart colors
BLUE = "#2a78d6"
ORANGE = "#eb6834"
GRID = "#e4e3df"

# default look for every matplotlib chart: no top/right border, light grid behind the bars
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": GRID,
    "axes.axisbelow": True,
})

#### Load

`dispatch_date_time` is stored in UTC. Philadelphia is 4-5 hours behind UTC, so it is converted to
local time and the year/month/hour/day-night columns are derived from that (this also keeps the
notebook correct for CSVs produced by older versions of the cleaning script).

In [ ]:
# load the cleaned data; psa is read as text because it mixes numbers and letters
df = pd.read_csv(CLEANED_CSV, dtype={"psa": "string"}, parse_dates=["dispatch_date_time"])

# convert UTC timestamps to Philadelphia local time
df["dispatch_date_time"] = df["dispatch_date_time"].dt.tz_convert(LOCAL_TZ)

# recompute the date parts from local time
df["year"] = df["dispatch_date_time"].dt.year
df["month"] = df["dispatch_date_time"].dt.month
df["hour"] = df["dispatch_date_time"].dt.hour
df["time"] = df["dispatch_date_time"].dt.time

# day = 06:00-17:59, night = everything else
df["is_day_night"] = df["hour"].between(6, 17).map({True: "day", False: "night"})

# column types and a first look at the rows
df.info()
df.head()

#### Overview

In [ ]:
# size of the dataset, time span covered and number of categories
print("Rows:", len(df))
print("Date range:", df["dispatch_date_time"].min(), "->", df["dispatch_date_time"].max())
print("Crime types:", df["text_general_code"].nunique())
print("Districts:", df["dc_dist"].nunique())

# missing values left after cleaning
print("\nNull values:\n", df.isnull().sum())

**Insights: overview**

- **3.55M incidents** from Jan 2006 to 28 Sep 2026 (2026 is a partial year), across **32 crime types**.
- `dc_dist` has **25 district codes**, but only 21 districts exist today: 4, 6, 23 and 92 were retired, and 77 is the airport.
- Missing values are negligible: `psa` (1,253 rows) and `location_block` (79). Timestamps and coordinates are complete because the cleaning script dropped rows without them.

#### Most common crime types

In [ ]:
# the 15 most frequent crime types
top_types = df["text_general_code"].value_counts().head(15)

# horizontal bar chart, largest at the top
ax = top_types.sort_values().plot.barh(color=BLUE, figsize=(10, 6))
ax.set_title("Top 15 crime types (all years)")
ax.set_xlabel("Incidents")
ax.set_ylabel("")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Insights: crime types**

- The distribution is very concentrated: the **top 5 types make up 64%** of all incidents and the top 15 make up 94%.
- The largest category, **All Other Offenses (17%)**, is a catch-all, so it says little on its own.
- **Property crime dominates.** Thefts (15%), Theft from Vehicle (8%), Vandalism (9%), Motor Vehicle Theft and Residential Burglary (4% each) together make up about 40%.
- Gun crimes are a small share by count: Robbery with a firearm 1.7% and Aggravated Assault with a firearm 1.5%. They still matter most for harm.

#### Incidents per year

The last year is partial, so its drop is not a real trend.

In [ ]:
# number of incidents in each year
per_year = df.groupby("year").size()

# line chart of the yearly trend, y axis starting at 0
ax = per_year.plot(color=BLUE, linewidth=2, marker="o", markersize=5)
ax.set_title("Incidents per year")
ax.set_xticks(per_year.index[::2])  # label every other year
ax.set_xlabel("")
ax.set_ylabel("Incidents")
ax.set_ylim(0)
plt.tight_layout()
plt.show()

**Insights: yearly trend**

- Recorded incidents fell **38%, from 217.8k (2006) to 134.3k (2021)**, with a steady decline over the 2010s.
- **2020-2021 are the lowest years**, which coincides with the COVID-19 pandemic.
- After a **rebound to 162k in 2023**, the count eased to 148.8k in 2025.
- **2026 only runs to 28 Sep.** Its 112.7k incidents put it on pace for roughly 150k, about the same as 2025.
- The mix of crimes changed a lot (average 2006-08 vs 2023-25):
  - **Narcotics (-82%), DUI (-84%) and Disorderly Conduct (-86%)** collapsed. These are mostly police-initiated, so the drop likely reflects enforcement priorities as much as behaviour.
  - **Motor Vehicle Theft (+167%) and Thefts (+72%)** rose sharply, while Other Assaults stayed flat.

#### Seasonality

Average incidents per calendar month, using complete years only.

In [ ]:
# keep only complete years so the partial current year doesn't skew the averages
last_full_year = df["year"].max() - 1
full_years = df[df["year"] <= last_full_year]

# count incidents per (year, month), then average each month across years
monthly = full_years.groupby(["year", "month"]).size().groupby("month").mean()

# bar chart, one bar per calendar month
ax = monthly.plot.bar(color=BLUE, rot=0)
ax.set_title(f"Average incidents per month ({full_years['year'].min()}-{last_full_year})")
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_xlabel("")
ax.set_ylabel("Incidents")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**Insights: seasonality**

- Crime is **seasonal: it peaks in summer** (May-Aug, about 15.3-15.7k per month) and is lowest in winter (Dec-Feb).
- February looks lowest mostly because it is shorter. Per day, it is **about 420-430 incidents in Dec-Feb vs about 505-510 in Jun-Aug**, roughly **20% more in summer**.

#### Time of day

Day is 06:00-17:59 local time, night is everything else.

In [ ]:
# number of incidents in each hour of the day (all years combined)
hourly = df.groupby("hour").size()

# color day hours blue and night hours orange
colors = [BLUE if 6 <= h < 18 else ORANGE for h in hourly.index]

fig, ax = plt.subplots()
ax.bar(hourly.index, hourly.values, color=colors)
ax.set_title("Incidents by hour of day (local time)")
ax.set_xlabel("Hour")
ax.set_ylabel("Incidents")
ax.set_xticks(range(0, 24, 2))
ax.grid(axis="x", visible=False)

# legend built by hand since the bars are a single series with two colors
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)],
          labels=["Day", "Night"], frameon=False)
plt.tight_layout()
plt.show()

# overall share of incidents during the day vs at night
print(df["is_day_night"].value_counts(normalize=True))

**Insights: time of day**

- The quietest time is **4-6 am** (about 1% of incidents per hour). Activity rises quickly from 7 am and plateaus around 10 am-1 pm.
- The busiest hour is **4 pm (6.5%)**, followed by 5 pm. There is a noticeable dip at 2 pm just before it.
- Activity stays high through the evening and only falls off after midnight.
- **55% of incidents happen during the day** (06:00-17:59) and 45% at night.
- Caveat: `dispatch_date_time` is when police were dispatched, not when the crime happened.

In [ ]:
# share of each top crime type that happens at night
night_share = (
    df[df["text_general_code"].isin(top_types.index)]
    .groupby("text_general_code")["is_day_night"]
    .apply(lambda s: (s == "night").mean())
    .sort_values()
)

# horizontal bars; the dashed line at 0.5 marks an even day/night split
ax = night_share.plot.barh(color=ORANGE, figsize=(10, 6))
ax.axvline(0.5, color="#52514e", linewidth=1, linestyle="--")
ax.set_title("Share of incidents at night, top 15 crime types")
ax.set_xlabel("Share at night")
ax.set_ylabel("")
ax.set_xlim(0, 1)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Insights: day vs night by crime type**

- **The most night-heavy types:**
  - **DUI (85% at night, peak 2 am)**
  - **Robbery with a firearm (68%, peak 11 pm)**
  - **Aggravated Assault with a firearm (61%, peak 10 pm)**

  Gun violence is concentrated in late evening.
- **The most day-heavy types:** Fraud (23% at night), Theft from Vehicle (27%), Motor Vehicle Theft (32%) and Thefts (32%).
- Vehicle-related thefts peak at **8 am**. That most likely reflects when owners *discover* and report them, not when they happen, which shows the dispatch-time caveat in action.
- Data quality: **about 74% of incidents stamped exactly 00:00 are homicides**, so homicide times look like placeholders and shouldn't be used for time-of-day analysis.

#### Police districts

In [ ]:
# number of incidents recorded under each police district code (dc_dist)
per_district = df["dc_dist"].astype(int).value_counts().sort_values(ascending=False)

# bar chart, busiest district first
ax = per_district.plot.bar(color=BLUE, rot=0)
ax.set_title("Incidents per police district (all years)")
ax.set_xlabel("District")
ax.set_ylabel("Incidents")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**Insights: districts (by `dc_dist` code)**

- The busiest codes are **15 (283k), 24 (257k), 25 (227k), 19 (226k) and 22 (224k)**. 15 covers a large part of the Northeast, and 24/25 cover the Kensington / lower North Philadelphia area.
- The quietest active districts are 5, 7 and 1, which are mostly residential, lower-density areas.
- These counts are **not comparable over time**, because the district map changed:
  - code 23 was used until 2013
  - code 92 until 2009
  - code 4 until 2023
  - code 6 until 2024

  Their incidents now belong to neighbouring districts. The choropleth below fixes this by assigning incidents with their coordinates.

#### Where incidents happen

Density of incident locations. Points outside Philadelphia's bounding box are dropped as bad coordinates.

In [ ]:
# keep only points inside a box around Philadelphia; anything outside is a bad coordinate
in_city = df["point_x"].between(-75.30, -74.95) & df["point_y"].between(39.86, 40.14)
print(f"Dropped {(~in_city).sum()} rows with coordinates outside the city")
pts = df[in_city]

# hexagon density map; log color scale so quieter areas are still visible
fig, ax = plt.subplots(figsize=(8, 8))
hb = ax.hexbin(pts["point_x"], pts["point_y"], gridsize=120, bins="log", cmap="Blues", mincnt=1)
fig.colorbar(hb, ax=ax, label="Incidents (log scale)", shrink=0.7)
ax.set_title("Incident density")
ax.set_aspect(1 / 0.77)  # ~cos(40°) so the map isn't stretched

# hide axes, grid and borders so it reads as a map
ax.set_xticks([])
ax.set_yticks([])
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.show()

**Insights: incident density**

- Only **375 rows** had coordinates outside the city, so location data is reliable.
- The darkest areas run from **Center City north through North Philadelphia and Kensington**.
- Activity thins out toward the outer Northeast and Northwest. Large gaps match non-residential land such as Fairmount Park, the rivers, the airport and the Navy Yard.

#### Choropleth by police district

District boundaries come from OpenDataPhilly. Districts 4, 6, 23 and 92 in the data no longer exist
(merged into neighbours), so each incident is assigned to a current district by its coordinates
instead of by `dc_dist`. Set `CRIME_TYPE` / `YEARS` to filter the map.

In [ ]:
import numpy as np
import plotly.express as px
import requests
from matplotlib.path import Path

# download the current police district boundaries (GeoJSON) from OpenDataPhilly
DISTRICTS_URL = "https://opendata.arcgis.com/datasets/62ec63afb8824a15953399b1fa819df2_0.geojson"
districts = requests.get(DISTRICTS_URL, timeout=60).json()

# assign every incident to the current district its coordinates fall in
xy = pts[["point_x", "point_y"]].to_numpy()
current_dist = np.full(len(pts), -1)  # -1 = not inside any district
for feature in districts["features"]:
    geom = feature["geometry"]
    # a district can be one polygon or several (MultiPolygon); handle both the same way
    polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    for polygon in polygons:
        inside = Path(polygon[0]).contains_points(xy)  # outer ring
        current_dist[inside] = int(feature["properties"]["dist_numc"])
pts = pts.assign(current_dist=current_dist)
print(f"{(current_dist == -1).sum()} incidents fell outside every district")

In [ ]:
# filters: change these and re-run the cell
CRIME_TYPE = None    # e.g. "Thefts", or None for all crime types
YEARS = (2006, 2026)  # inclusive range

# apply the filters and drop incidents that weren't matched to a district
subset = pts[pts["year"].between(*YEARS) & (pts["current_dist"] != -1)]
if CRIME_TYPE:
    subset = subset[subset["text_general_code"] == CRIME_TYPE]

# incidents per district; district ids are strings to match the GeoJSON's dist_numc
counts = subset.groupby("current_dist").size().rename("incidents").reset_index()
counts["district"] = counts["current_dist"].astype(str)

# interactive choropleth: darker blue = more incidents, hover a district for its count
title = f"{CRIME_TYPE or 'All incidents'} by police district, {YEARS[0]}-{YEARS[1]}"
fig = px.choropleth_map(
    counts,
    geojson=districts,
    locations="district",
    featureidkey="properties.dist_numc",  # GeoJSON field that holds the district number
    color="incidents",
    color_continuous_scale="Blues",
    hover_name="district",
    hover_data={"district": False, "incidents": ":,"},
    map_style="carto-positron",           # light street basemap, no API key needed
    center={"lat": 40.0, "lon": -75.13},
    zoom=9.6,
    opacity=0.8,
    title=title,
)
fig.update_layout(height=650, margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig.show()

**Insights: choropleth**

- After assigning incidents to *current* boundaries, **District 9 tops the map (304k)** because it absorbed old District 6 (Center City). It is followed by **15 (284k), 24 (259k) and 22 (248k)**.
- District 77 (the airport) has just 4k.
- **8,959 incidents (0.25%)** fall outside every district polygon, mostly points on rivers or right on boundary edges.
- The map shows **raw counts**, so large districts look worse partly because of their size. For a fairer comparison, normalise by area or population.

### Summary

1. **Crime has fallen long-term.** It dropped 38% from 2006 to the 2021 low, then partly rebounded. The mix shifted from enforcement-driven offences (narcotics, DUI) toward property crime, especially vehicle theft.
2. **It is strongly seasonal and daily.** There are about 20% more incidents per day in summer than in winter. The peak is late afternoon (4-5 pm) and the low is 4-6 am.
3. **Crime types have distinct time signatures.** Gun violence and DUI happen late at night, while thefts and fraud show up during the day, partly because of when they are discovered.
4. **It is geographically concentrated.** It centres on Center City and North Philadelphia / Kensington (districts 9, 15, 24, 22, 25).

**Data caveats for later steps:**
- **Time zone:** `dispatch_date_time` is UTC; always convert to Philadelphia time before deriving hours or dates.
- **Retired districts:** 4, 6, 23 and 92 are no longer in use, so district-level trends need current boundaries.
- **Homicide times:** these are mostly placeholder 00:00 timestamps.
- **Dispatch time:** it reflects when police were dispatched, not when the crime happened.